# Multiclass Data Preparation & Curation Pipeline

This notebook covers the data preparation and label engineering pipeline for **Multiclass and Multi-Model Classifiers**. It executes the following stages:

1. **Baseline Multiclass Dataset (`multiclass_df.csv`)**: Consolidates clinical self-reports, general controls, and suicidewatch posts into a unified dataframe mapping subreddits to diagnostic labels.
2. **Refined Multiclass Dataset (`multiclass_df_v2.csv`)**: Performs label alignment, maps minor/irrelevant classes to control, filters noise, and ensures class label consistency.
3. **Length Augmentation (`multiclass_df_augmented.csv`)**: Generates short and medium-length sub-samples of long posts to improve model robustess across different text lengths.
4. **Control Source Curation**: Analyzes and filters auxiliary emotion datasets (GoEmotions, Empathetic Dialogues, etc.) to construct curated control class samples.

---

## Setup & Global Imports

In [1]:
import pandas as pd
import numpy as np
import re

## Section 1: Baseline Multiclass Dataset Generation (`multiclass_df.csv`)
Consolidates three datasets (third_dataset, self_reported_df, and suicidewatch controls) into a single dataset with basic multi-class and three-class labels.

In [2]:
# Load source datasets
third_dataset = pd.read_csv("third_dataset.csv")
self_reported = pd.read_csv("self_reported_df.csv")
mental_disorders_subreddits_control = pd.read_csv("mental_disorders_subreddits_control.csv")

# Separate controls and suicidewatch
control = third_dataset[third_dataset['label'] == 0]
suicidal = mental_disorders_subreddits_control[mental_disorders_subreddits_control['subreddit'] == 'suicidewatch'].copy()
suicidal['label'] = 1

# Concat datasets and drop duplicates
multiclass_df = pd.concat([self_reported, third_dataset, suicidal], ignore_index=True)
if 'Unnamed: 0' in multiclass_df.columns:
    multiclass_df = multiclass_df.drop(columns=['Unnamed: 0'])
multiclass_df = multiclass_df.drop_duplicates().reset_index(drop=True)

print("Concatenated Multiclass Shape:", multiclass_df.shape)
print("Subreddit Counts:\n", multiclass_df["subreddit"].value_counts())

C:\Users\hana\AppData\Local\Temp\ipykernel_19656\2666168447.py:4: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  mental_disorders_subreddits_control = pd.read_csv("mental_disorders_subreddits_control.csv")


Concatenated Multiclass Shape: (81581, 11)
Subreddit Counts:
 subreddit
suicidewatch       15033
relationships      14234
adhd                8328
anxiety             6914
depression          6488
mentalhealth        5326
conspiracy          4615
lonely              3316
meditation          2554
socialanxiety       2020
bpd                 1968
divorce             1844
ptsd                1768
jokes               1163
autism              1144
schizophrenia        920
healthanxiety        836
bipolarreddit        700
edanonymous          636
addiction            346
fitness              263
legaladvice          258
alcoholism           256
personalfinance      186
covid19_support      172
parenting            158
guns                 111
teaching              24
Name: count, dtype: int64


### 1.2 Define Diagnostic Labels
Create `multiclass_label` and `threeclass_label` columns based on the post's origin subreddit.

In [3]:
# Map subreddit names to multiclass diagnostic labels (defaulting label 0 / controls to 'control')
multiclass_df['multiclass_label'] = multiclass_df['subreddit'].where(
    multiclass_df['label'] == 1,
    'control'
)

# Shuffle the dataset
multiclass_df = multiclass_df.sample(frac=1, random_state=42).reset_index(drop=True)
print("Multiclass Label distribution:\n", multiclass_df['multiclass_label'].value_counts())

# Map multiclass labels to threeclass severity level labels (normal, at_risk, suicidal)
multiclass_df["threeclass_label"] = (
    multiclass_df["multiclass_label"]
    .map({
        "control": "normal",
        "suicidewatch": "suicidal"
    })
    .fillna("at_risk")
)

# Save baseline multiclass dataset
multiclass_df.to_csv("multiclass_df.csv", index=False)
print("Baseline Multiclass Dataset Saved! Shape:", multiclass_df.shape)

Multiclass Label distribution:
 multiclass_label
control            45938
suicidewatch       14369
adhd                4164
anxiety             3457
depression          3244
mentalhealth        2663
relationships       1639
socialanxiety       1010
bpd                  984
ptsd                 884
autism               572
schizophrenia        460
healthanxiety        418
bipolarreddit        350
edanonymous          318
lonely               295
meditation           241
divorce              178
addiction            173
alcoholism           128
conspiracy            53
covid19_support       37
jokes                  6
Name: count, dtype: int64
Baseline Multiclass Dataset Saved! Shape: (81581, 13)


## Section 2: Label Refinement & Class Curation (`multiclass_df_v2.csv`)
Refine class distributions by dropping noisy classes, mapping non-clinical/irrelevant subreddits (e.g., relationship support, jokes) back to controls, and aligning clinical labels.

In [4]:
multiclass_df = pd.read_csv("multiclass_df.csv")

# 1. Drop noisy or unsupported minor classes
drop_classes = ['covid19_support']
before = len(multiclass_df)
multiclass_df = multiclass_df[~multiclass_df['multiclass_label'].isin(drop_classes)].reset_index(drop=True)
print(f'Dropped {before - len(multiclass_df)} rows containing classes {drop_classes}')

# 2. Relabel non-clinical support subreddits to control
relabel_to_control = ['relationships', 'divorce', 'jokes', 'conspiracy', 'meditation']
mask = multiclass_df['multiclass_label'].isin(relabel_to_control)
multiclass_df.loc[mask, 'multiclass_label'] = 'control'
multiclass_df.loc[mask, 'threeclass_label'] = 'normal'
print(f'Relabeled {mask.sum()} rows to control/normal')

Dropped 37 rows containing classes ['covid19_support']
Relabeled 2117 rows to control/normal


### 2.2 Re-aligning Mislabelled Controls
Ensure posts from clinical subreddits that were misclassified or marked as controls are restored to their actual clinical labels.

In [5]:
classes_should_not_be_control = [
    "adhd", "anxiety", "depression", "bipolarreddit", "schizophrenia", 
    "ptsd", "autism", "suicidewatch", "healthanxiety", "edanonymous",
    "addiction", "alcoholism", "bpd", "socialanxiety", 'mentalhealth'
]

# Restore multiclass diagnostic label
multiclass_df.loc[
    multiclass_df['subreddit'].isin(classes_should_not_be_control) & (multiclass_df['multiclass_label'] == 'control'), 
    'multiclass_label'
] = multiclass_df['subreddit']

# Restore suicidal threeclass label
multiclass_df.loc[
    (multiclass_df['subreddit'] == 'suicidewatch') & (multiclass_df['threeclass_label'] == 'normal'),
    'threeclass_label'
] = 'suicidal'

# Restore at-risk threeclass label
multiclass_df.loc[
    (multiclass_df['subreddit'].isin(classes_should_not_be_control)) &
    (multiclass_df['subreddit'] != 'suicidewatch') &
    (multiclass_df['threeclass_label'] == 'normal'),
    'threeclass_label'
] = 'at_risk'

# Drop remaining control items from covid19_support
multiclass_df = multiclass_df[
    ~((multiclass_df['subreddit'] == "covid19_support") & (multiclass_df['multiclass_label'] == 'control'))
]

print("Updated Multiclass distribution:\n", multiclass_df['multiclass_label'].value_counts())

Updated Multiclass distribution:
 multiclass_label
control          28431
suicidewatch     15033
adhd              8328
anxiety           6914
depression        6488
mentalhealth      5326
socialanxiety     2020
bpd               1968
ptsd              1768
autism            1144
schizophrenia      920
healthanxiety      836
bipolarreddit      700
edanonymous        636
addiction          346
lonely             295
alcoholism         256
Name: count, dtype: int64


### 2.3 Curation of Self-Report Labels & Dropping Unwanted Subreddits
Mark posts with explicit self-reports as `at_risk`, filter noisy subreddits (like lonely) and drop irrelevant at-risk categories.

In [6]:
# Mark self-reports as at-risk
multiclass_df.loc[multiclass_df['is_self_report'] == 1, 'threeclass_label'] = 'at_risk'

# Exclude general 'lonely' posts from control
multiclass_df = multiclass_df[
    ~((multiclass_df['subreddit'] == "lonely") & (multiclass_df['multiclass_label'] == 'control'))
]

# Drop unwanted at-risk subreddits to clean decision boundary
unwanted_subreddits_at_risk = ['meditation', 'conspiracy', 'jokes', 'parenting', 'legaladvice', 'personalfinance', 'fitness']
multiclass_df.drop(multiclass_df[
    (multiclass_df['threeclass_label'] == 'at_risk') &
    (multiclass_df['subreddit'].isin(unwanted_subreddits_at_risk))
].index, inplace=True)

# Ensure suicidewatch posts are mapped to 'suicidal'
multiclass_df.loc[multiclass_df['subreddit'] == 'suicidewatch', 'threeclass_label'] = 'suicidal'

# Save refined multiclass dataset
multiclass_df.to_csv("multiclass_df_v2.csv", index=False)
print("Refined Multiclass Dataset Saved! Shape:", multiclass_df.shape)

Refined Multiclass Dataset Saved! Shape: (77781, 13)


## Section 3: Length Augmentation (`multiclass_df_augmented.csv`)
To improve model performance across different text lengths, we generate shorter versions of long posts:
- For posts > 150 words: add a **short** version (first 50 words).
- For posts > 300 words: add a **medium** version (first 150 words).
All original rows are kept intact.

In [7]:
df_v2 = pd.read_csv('multiclass_df_v2.csv')
print(f'Original shape: {df_v2.shape}')

def front_truncate(text, n_words):
    """Return the first n_words words of text."""
    if not isinstance(text, str):
        return ''
    words = text.split()
    return ' '.join(words[:n_words])

SHORT_W  = 50
MEDIUM_W = 150

# Pre-compute word counts
df_v2['_wc'] = df_v2['other_posts'].apply(lambda x: len(str(x).split()))

# Buckets based on length
df_v2['_bucket'] = 'original'

extra_rows = []
for idx, row in df_v2.iterrows():
    wc = row['_wc']
    if wc > 150:
        # Create short version
        row_short = row.copy()
        row_short['other_posts'] = front_truncate(row['other_posts'], SHORT_W)
        row_short['_bucket'] = 'short'
        extra_rows.append(row_short)
        
    if wc > 300:
        # Create medium version
        row_medium = row.copy()
        row_medium['other_posts'] = front_truncate(row['other_posts'], MEDIUM_W)
        row_medium['_bucket'] = 'medium'
        extra_rows.append(row_medium)

if extra_rows:
    augmented = pd.concat([df_v2, pd.DataFrame(extra_rows)], ignore_index=True)
else:
    augmented = df_v2.copy()

# Shuffle and clean up helper columns
augmented = augmented.sample(frac=1, random_state=42).reset_index(drop=True)
print("Length Augmentation Statistics:\n", augmented['_bucket'].value_counts())

augmented = augmented.drop(columns=['_wc', '_bucket'])
augmented.to_csv('multiclass_df_augmented.csv', index=False)
print('Saved length augmented dataset -> multiclass_df_augmented.csv')

Original shape: (77781, 13)
Length Augmentation Statistics:
 _bucket
original    77781
short       47810
medium      27406
Name: count, dtype: int64
Saved length augmented dataset -> multiclass_df_augmented.csv


## Section 4: Auxiliary Emotion Datasets Curation
This section inspects and curates auxiliary datasets to gather healthy, expressive, and optimistic posts to populate the control/normal class.

### 4.1 GoEmotions Dataset

In [8]:
go_emotions = pd.read_csv('go_emotions_dataset.csv')

positive = ['admiration', 'amusement', 'approval', 'caring', 'curiosity', 'desire',
            'excitement', 'gratitude', 'joy', 'love', 'optimism', 'pride', 'relief']

negative = ['anger', 'annoyance', 'confusion', 'disappointment', 'disapproval',
            'disgust', 'embarrassment', 'fear', 'grief', 'nervousness', 'remorse', 'sadness']

go_emotions['positive_score'] = go_emotions[positive].sum(axis=1)
go_emotions['negative_score'] = go_emotions[negative].sum(axis=1)

# Classify dominant sentiment
go_emotions['sentiment'] = go_emotions.apply(
    lambda row: 'positive' if row['positive_score'] > row['negative_score']
                else ('negative' if row['negative_score'] > row['positive_score']
                      else 'neutral'),
    axis=1
)
print("GoEmotions Sentiment counts:\n", go_emotions['sentiment'].value_counts())

GoEmotions Sentiment counts:
 sentiment
positive    83277
neutral     74031
negative    53917
Name: count, dtype: int64


### 4.2 Holistix & CBET Datasets

In [9]:
CBET = pd.read_csv("CBET.csv")
print("CBET Shape:", CBET.shape)

# Holistix labels: 5 corresponds to 'Emotional Aspect'
Holistix = pd.read_csv("Holistix.csv")
print("Holistix Shape:", Holistix.shape)
print("Emotional items count:", len(Holistix[Holistix["labels"] == 5]))

CBET Shape: (81163, 11)
Holistix Shape: (1412, 4)
Emotional items count: 221


### 4.3 Emotion Parquet Curation

In [11]:
emotion = pd.read_parquet("emotion.parquet")
map_emotion = {
    0: 'sadness',
    1: 'joy',
    2: 'love',
    3: 'anger',
    4: 'fear',
    5: 'surprise'
}
emotion['emotion_label'] = emotion['label'].map(map_emotion)

# Select positive emotions for control data
positive_emotions = emotion[emotion['emotion_label'].isin(['joy', 'love'])]
print("Positive Emotion Samples:", len(positive_emotions))

Positive Emotion Samples: 175621


### 4.4 Empathetic Dialogues Curation

In [12]:
empatheticdialogues = pd.read_csv("train_ED.csv", engine="python", on_bad_lines="skip")
print("Empathetic Dialogues Context Counts:\n", empatheticdialogues['context'].value_counts())

positive_context = [
    'surprised', 'excited', 'proud', 'grateful', 
    'confident', 'hopeful', 'impressed', 'nostalgic', 
    'joyful', 'prepared', 'content', 'sentimental',
    'caring', 'trusting', 'faithful'
]

negative_context = [
    'angry', 'annoyed', 'sad', 'afraid',
    'lonely', 'terrified', 'anxious', 'guilty',
    'disgusted', 'disappointed', 'furious', 
    'devastated', 'embarrassed', 'jealous',
    'ashamed', 'apprehensive'
]

# Filter and remove duplicates (fixing the original notebook's inplace drop_duplicates bugs)
positive_context_df = empatheticdialogues[empatheticdialogues['context'].isin(positive_context)].copy()
positive_context_df = positive_context_df.drop_duplicates(subset=['prompt']).reset_index(drop=True)
print("Empathetic Dialogues Positive Prompts Shape:", positive_context_df.shape)

Empathetic Dialogues Context Counts:
 context
surprised       3949
excited         2927
angry           2734
proud           2709
annoyed         2635
sad             2630
afraid          2504
lonely          2494
terrified       2487
grateful        2474
anxious         2453
guilty          2452
disgusted       2447
anticipating    2438
confident       2433
hopeful         2396
furious         2382
impressed       2378
disappointed    2350
nostalgic       2349
joyful          2341
jealous         2324
prepared        2290
content         2212
devastated      2191
embarrassed     2183
sentimental     2068
caring          2051
trusting        1994
ashamed         1967
apprehensive    1815
faithful        1435
Name: count, dtype: int64
Empathetic Dialogues Positive Prompts Shape: (8267, 8)
